# Análise de Sentimentos: comparação de três métodos em teste reservado

**Continuação de** [`sentimento_01_validacao_amostra_anotada.ipynb`](sentimento_01_validacao_amostra_anotada.ipynb).
**Arquivos:** pasta [`validacao_sentimento/`](validacao_sentimento/). **ADR:** 0048.

## 1. Pergunta

A primeira validação mostrou que o classificador lê como `neutral` o apoio em Código de campanha e motivou uma regra de
pós-processamento. Há uma alternativa de fundo: ajustar o próprio classificador com exemplos anotados. Este notebook compara os três
métodos **no mesmo conjunto de teste**, que nenhum deles viu, e escolhe um por um critério fixado antes de olhar o resultado.

| Método | O que é |
|---|---|
| **Original** | `cardiffnlp/twitter-xlm-roberta-base-sentiment`, sem ajuste |
| **Regra** | Original + regra do Código de campanha (`src/modeling/sentiment_rules.py`), congelada antes do sorteio do teste |
| **Ajuste fino** | Original com o treino continuado em comentários anotados (`validacao_sentimento/ajuste_fino.py`) |

## 2. Protocolo contra vazamento de dados

| Risco | Como foi evitado |
|---|---|
| A regra ser avaliada na amostra em que foi desenhada | O teste é um sorteio **novo**. Os comentários das amostras da primeira validação não podem ser sorteados para ele |
| O modelo ajustado ser avaliado em textos que viu no treino | Todo comentário cujo **texto normalizado** aparece no teste é retirado do treino. A normalização ignora caixa, espaços, tom de pele e repetição de caractere, de modo que "👏👏👏" e "👏👏👏👏" contam como o mesmo texto |
| Comentários da mesma publicação no treino e no teste (mesmo assunto, mesmo vocabulário) | Toda publicação com comentário no teste é retirada inteira do treino |
| Escolher época e taxa de aprendizado olhando o teste | A escolha é feita no conjunto de **desenvolvimento** (a amostra da primeira validação). O script de treino não lê os rótulos do teste |
| Textos do desenvolvimento no treino (inflaria a escolha da época) | Os textos normalizados do desenvolvimento também são retirados do treino |
| Anotar o teste já sabendo o que os métodos responderam | O teste foi anotado só com o texto, **antes** de qualquer treino e sem o rótulo de nenhum método; o arquivo de rótulos tem o resumo criptográfico registrado abaixo |
| Rodar o teste várias vezes e ficar com a melhor | As previsões do modelo ajustado no teste são geradas uma única vez, depois de a configuração estar escolhida |
| Escolher o vencedor depois de ver os números | O critério de escolha está escrito na Seção 6, antes dos resultados |

O que **não** foi possível evitar: quem anotou o teste é o mesmo modelo de linguagem que desenhou a regra e escreveu o critério de
anotação, e os rótulos de treino foram anotados por outras instâncias do mesmo modelo, com o mesmo critério. O teste mede,
portanto, a concordância de cada método com esse critério; a conferência do autor na planilha é o que o ancora em julgamento humano.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve().parents[1]
PASTA = Path.cwd().resolve() / "validacao_sentimento"
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(PASTA))

import amostragem as am
import particao
from src.modeling.sentiment_rules import is_codigo_de_campanha
from src.repositories.delta_repository import DeltaRepository

pd.set_option("display.max_colwidth", 200)
CLASSES = am.CLASSES
METODOS = ["original", "regra", "ajuste fino"]
repo = DeltaRepository(RAIZ / "data" / "gold", RAIZ / "data" / "silver", RAIZ / "data" / "bronze")
pop = am.carregar_populacao(repo)   # `sentiment_label` aqui é o rótulo do classificador original
W = pop["sentiment_label"].value_counts(normalize=True).reindex(CLASSES).to_numpy()

## 3. Partição

O sorteio está em [`validacao_sentimento/particao.py`](validacao_sentimento/particao.py). A célula refaz o sorteio do teste e do
treino e confere que eles reproduzem os arquivos anotados; depois confere, uma a uma, as condições de não vazamento.

In [2]:
teste_sorteado = particao.sortear_teste(pop)
teste_gab = pd.read_csv(particao.ARQ_TESTE_GABARITO)
teste_txt = pd.read_csv(particao.ARQ_TESTE)
assert list(teste_sorteado["id_comment"].astype(str)) == list(teste_gab["id_comment"].astype(str)), "o sorteio não reproduz o teste"

treino_sorteado = particao.sortear_treino(pop, teste_sorteado)
treino_txt = pd.read_csv(particao.ARQ_TREINO)
assert list(treino_sorteado["id_comment"].astype(str)) == list(treino_txt["id_comment"].astype(str)), "o sorteio não reproduz o treino"

dev_txt = pd.read_csv(am.ARQ_AMOSTRA)
chave = particao.chave_texto
chaves = {nome: set(df["text"].map(chave)) for nome, df in (("treino", treino_txt), ("desenvolvimento", dev_txt), ("teste", teste_txt))}

conferencias = pd.Series({
    "comentários do teste que estavam nas amostras da primeira validação": len(set(teste_gab["id_comment"].astype(str)) & particao.ids_da_primeira_validacao()),
    "textos normalizados do treino que aparecem no teste": len(chaves["treino"] & chaves["teste"]),
    "textos normalizados do treino que aparecem no desenvolvimento": len(chaves["treino"] & chaves["desenvolvimento"]),
    "publicações do treino com comentário no teste": len(set(treino_txt["id_reel"].astype(str)) & set(teste_gab["id_reel"].astype(str))),
    "textos normalizados repetidos dentro do treino": int(treino_txt["text"].map(chave).duplicated().sum()),
}, name="ocorrências")
assert (conferencias == 0).all(), conferencias
display(conferencias.to_frame())

tamanhos = pd.DataFrame({
    "linhas": [len(treino_txt), len(dev_txt), len(teste_txt)],
    "textos normalizados distintos": [len(chaves["treino"]), len(chaves["desenvolvimento"]), len(chaves["teste"])],
    "papel": ["ajustar o modelo", "escolher época e taxa de aprendizado", "comparar os três métodos, uma vez"],
}, index=["treino", "desenvolvimento", "teste"])
display(tamanhos)
print("Teste por classe do classificador original:", teste_gab["sentiment_label"].value_counts().reindex(CLASSES).to_dict())
print("Textos normalizados do teste que também estão no desenvolvimento (permitido: o teste é um sorteio da população):", len(chaves["teste"] & chaves["desenvolvimento"]))

,ocorrências
comentários do teste que estavam nas amostras da primeira validação,0
textos normalizados do treino que aparecem no teste,0
textos normalizados do treino que aparecem no desenvolvimento,0
publicações do treino com comentário no teste,0
textos normalizados repetidos dentro do treino,0


,linhas,textos normalizados distintos,papel
treino,1482,1482,ajustar o modelo
desenvolvimento,360,288,escolher época e taxa de aprendizado
teste,450,368,"comparar os três métodos, uma vez"


Teste por classe do classificador original: {'positive': 150, 'neutral': 150, 'negative': 150}
Textos normalizados do teste que também estão no desenvolvimento (permitido: o teste é um sorteio da população): 29


**Consequência da exclusão por texto.** Os textos mais comuns da população (aplausos, corações, os códigos de campanha mais
frequentes) aparecem no teste e, por isso, ficam fora do treino. O modelo ajustado é avaliado pela capacidade de **generalizar** a
partir de variações, não de reconhecer textos decorados. É uma condição mais dura do que a de uso real, em que os mesmos textos se
repetem, e por isso o resultado do ajuste fino aqui é conservador.

## 4. Anotação

Mesmo critério da primeira validação, em [`validacao_sentimento/criterio_anotacao.md`](validacao_sentimento/criterio_anotacao.md).

| Conjunto | Quem anotou | Conferência |
|---|---|---|
| Teste (450) | Modelo de linguagem (Claude, da Anthropic), só com o texto, antes do treino | Autor, em `teste_planilha_conferencia.xlsx` (sem o rótulo de nenhum método) |
| Treino | Quatro instâncias independentes do mesmo modelo de linguagem, um quarto dos textos cada, só com o texto e o critério | Não conferido: é dado de treino, e o erro de anotação ali custa desempenho, não validade |
| Desenvolvimento (360) | A anotação da primeira validação | Autor, em `planilha_conferencia.xlsx` |

In [3]:
ARQ_ROTULOS_TESTE = PASTA / "teste_rotulos_anotador.csv"
print("SHA-256 dos rótulos originais do teste:", hashlib.sha256(ARQ_ROTULOS_TESTE.read_bytes()).hexdigest())
rot_teste = pd.read_csv(ARQ_ROTULOS_TESTE, keep_default_na=False)

ARQ_PLANILHA_TESTE = PASTA / "teste_planilha_conferencia.xlsx"
if not ARQ_PLANILHA_TESTE.exists():
    p = teste_txt.merge(rot_teste, on="id_amostra").rename(columns={"rotulo": "rotulo_anotador"})
    p["rotulo_revisado"] = ""
    p["comentario_revisor"] = ""
    with pd.ExcelWriter(ARQ_PLANILHA_TESTE, engine="openpyxl") as xl:
        p.to_excel(xl, index=False, sheet_name="conferencia")
        for coluna, largura in zip("ABCDEF", (11, 100, 16, 20, 16, 40)):
            xl.sheets["conferencia"].column_dimensions[coluna].width = largura
        xl.sheets["conferencia"].freeze_panes = "A2"
p = pd.read_excel(ARQ_PLANILHA_TESTE, dtype=str, keep_default_na=False)
rev = p["rotulo_revisado"].str.strip().str.lower()
assert rev[rev != ""].isin(CLASSES).all()
p["real"] = np.where(rev != "", rev, p["rotulo_anotador"])
print(f"Rótulos do teste conferidos: {int((rev != '').sum())} de {len(p)}; alterados: {int(((rev != '') & (rev != p['rotulo_anotador'])).sum())}")

rot_treino = pd.read_csv(PASTA / "treino_rotulos_anotador.csv")
display(pd.DataFrame({
    "treino": rot_treino["rotulo"].value_counts().reindex(CLASSES),
    "teste (amostra estratificada)": p["real"].value_counts().reindex(CLASSES),
}))

SHA-256 dos rótulos originais do teste: 22da5357585911c8314f770f992a3853ecf344fae996a1758fab62a82b14091a
Rótulos do teste conferidos: 0 de 450; alterados: 0


,treino,teste (amostra estratificada)
positive,827,257
neutral,261,67
negative,394,126


## 5. Ajuste fino

O treino roda fora do notebook (`python notebooks/modelagens/validacao_sentimento/ajuste_fino.py`, em CPU) e grava o registro e as
previsões que este notebook lê. Configuração: parte dos pesos do classificador original; perda de entropia cruzada ponderada pelo
inverso da raiz da frequência de cada classe no treino; AdamW com aquecimento linear e decaimento; 64 *tokens* por texto. A cada
época o modelo é avaliado no desenvolvimento pelo F1-macro ponderado pela população, e fica a época de maior valor.

In [4]:
log = json.loads((PASTA / "ajuste_fino_log.json").read_text(encoding="utf-8"))
print({k: log[k] for k in ("modelo_base", "semente", "max_tokens", "lote", "epocas", "taxas_testadas", "n_treino", "distribuicao_treino", "minutos")})
print("Conferência de vazamento feita pelo script antes de treinar:", log["conferencia_de_vazamento"])
print(f"F1-macro no desenvolvimento: classificador original {log['f1_macro_dev_modelo_original']:.3f}; escolhido {log['escolhido']}")
pd.DataFrame(log["historico"]).pivot(index="epoca", columns="taxa", values=["perda_treino", "f1_macro_dev"]).style.format("{:.3f}")

{'modelo_base': 'cardiffnlp/twitter-xlm-roberta-base-sentiment', 'semente': 20261119, 'max_tokens': 64, 'lote': 16, 'epocas': 4, 'taxas_testadas': [1e-05, 3e-05], 'n_treino': 1482, 'distribuicao_treino': {'positive': 827, 'negative': 394, 'neutral': 261}, 'minutos': 93.7}
Conferência de vazamento feita pelo script antes de treinar: {'textos_do_treino_no_teste': 0, 'textos_do_treino_no_desenvolvimento': 0, 'publicacoes_do_treino_no_teste': 0, 'comentarios_do_teste_no_desenvolvimento': 0}
F1-macro no desenvolvimento: classificador original 0.654; escolhido {'taxa': 3e-05, 'epoca': 2, 'f1_macro_dev': 0.914255108203347}


## 6. Critério de escolha (fixado antes de ver o teste)

1. **Métrica principal:** F1-macro no teste, estimado para a população (cada estrato do teste entra com o peso da sua classe do
   classificador original na população). O F1-macro dá o mesmo peso às três classes, e a classe `negative`, que sustenta o achado
   central do TCC, é a menos frequente.
2. **Um método só vence o outro se a diferença for distinguível de zero:** o intervalo de 95% da diferença de F1-macro, por
   *bootstrap* pareado e estratificado (as mesmas reamostragens para os dois métodos), não pode incluir zero.
3. **Em empate, fica o mais simples:** original, depois regra, depois ajuste fino. O ajuste fino exige reexecutar o classificador em
   todo o corpus, publicar um modelo e abre mão de uma correção que cabe em uma frase.
4. **Salvaguarda da classe `negative`:** um método não é escolhido se o F1 de `negative` for pior que o do original com diferença
   distinguível de zero.

## 7. Resultado no teste

In [5]:
teste = p[["id_amostra", "text", "real"]].merge(teste_gab[["id_amostra", "sentiment_label", "inputUrl"]], on="id_amostra")
teste = teste.merge(pd.read_csv(PASTA / "ajuste_fino_previsoes_teste.csv")[["id_amostra", "previsto_ajuste_fino"]], on="id_amostra", validate="one_to_one")
teste["codigo_campanha"] = teste["text"].map(is_codigo_de_campanha)
teste["original"] = teste["sentiment_label"]
teste["regra"] = np.where((teste["original"] == "neutral") & teste["codigo_campanha"], "positive", teste["original"])
teste["ajuste fino"] = teste["previsto_ajuste_fino"]
assert len(teste) == 450

estrato_idx = teste["original"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
codigo = {c: i for i, c in enumerate(CLASSES)}
real_idx = teste["real"].map(codigo).to_numpy()
prev_idx = {m: teste[m].map(codigo).to_numpy() for m in METODOS}

def J_de(prev, real, estrato):
    J = np.zeros((3, 3))
    for i in range(3):
        sel = estrato == i
        np.add.at(J, (prev[sel], real[sel]), W[i] / sel.sum())
    return J

def metricas(J):
    with np.errstate(invalid="ignore", divide="ignore"):
        prec = np.diag(J) / J.sum(axis=1)
        rev = np.diag(J) / J.sum(axis=0)
        f1 = np.nan_to_num(2 * prec * rev / (prec + rev))
    return {"acurácia": np.trace(J), "F1-macro": f1.mean(), **{f"precisão {c}": prec[i] for i, c in enumerate(CLASSES)},
            **{f"revocação {c}": rev[i] for i, c in enumerate(CLASSES)}, **{f"F1 {c}": f1[i] for i, c in enumerate(CLASSES)},
            "positivos previstos": J.sum(axis=1)[0], "negativos previstos": J.sum(axis=1)[2]}

pontual = {m: metricas(J_de(prev_idx[m], real_idx, estrato_idx)) for m in METODOS}
real_J = J_de(prev_idx["original"], real_idx, estrato_idx).sum(axis=0)

rng = np.random.default_rng(am.SEED)
indices_por_estrato = [np.flatnonzero(estrato_idx == i) for i in range(3)]
boot = {m: [] for m in METODOS}
for _ in range(4000):
    re = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in indices_por_estrato])
    for m in METODOS:
        boot[m].append(metricas(J_de(prev_idx[m][re], real_idx[re], estrato_idx[re])))
boot = {m: pd.DataFrame(v) for m, v in boot.items()}

def com_ic(m, col):
    lo, hi = np.percentile(boot[m][col], [2.5, 97.5])
    return f"{pontual[m][col]:.1%} [{lo:.1%}; {hi:.1%}]"

resumo = pd.DataFrame({m: {col: com_ic(m, col) for col in pontual[m]} for m in METODOS})
print(f"Proporções estimadas pela anotação no teste: positivos {real_J[0]:.1%}, neutros {real_J[1]:.1%}, negativos {real_J[2]:.1%}")
resumo

Proporções estimadas pela anotação no teste: positivos 80.2%, neutros 10.8%, negativos 8.9%


,original,regra,ajuste fino
acurácia,74.7% [72.0%; 77.4%],86.9% [84.0%; 89.4%],93.9% [91.7%; 96.0%]
F1-macro,65.7% [61.2%; 70.1%],74.5% [69.6%; 79.4%],85.1% [80.2%; 89.6%]
precisão positive,98.0% [95.3%; 100.0%],98.3% [96.1%; 100.0%],97.3% [95.4%; 99.0%]
precisão neutral,28.7% [21.3%; 36.0%],47.3% [37.1%; 57.4%],86.3% [75.5%; 95.0%]
precisão negative,74.7% [67.3%; 81.3%],74.7% [67.3%; 81.3%],73.8% [63.2%; 83.7%]
revocação positive,74.7% [72.5%; 77.2%],89.9% [87.3%; 92.4%],97.8% [96.2%; 99.1%]
revocação neutral,81.9% [71.2%; 90.7%],81.9% [71.2%; 90.7%],72.4% [60.3%; 83.7%]
revocação negative,65.4% [54.9%; 78.6%],65.4% [54.9%; 78.6%],85.0% [73.7%; 95.0%]
F1 positive,84.8% [82.9%; 86.7%],94.0% [92.2%; 95.5%],97.6% [96.3%; 98.7%]
F1 neutral,42.5% [33.6%; 50.8%],59.9% [49.9%; 68.9%],78.7% [69.0%; 86.6%]


### 7.1 Diferenças pareadas

Diferença entre métodos na mesma reamostragem. Um intervalo que não inclui zero indica diferença distinguível.

In [6]:
def diferenca(a, b, col):
    d = boot[a][col] - boot[b][col]
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"comparação": f"{a} − {b}", "métrica": col, "diferença": pontual[a][col] - pontual[b][col], "IC 95% inf": lo, "IC 95% sup": hi,
            "distinguível de zero": bool(lo > 0 or hi < 0)}

pares = [("regra", "original"), ("ajuste fino", "original"), ("ajuste fino", "regra")]
dif = pd.DataFrame([diferenca(a, b, col) for col in ("F1-macro", "acurácia", "F1 negative", "F1 positive", "F1 neutral") for a, b in pares])
dif.style.format({"diferença": "{:+.1%}", "IC 95% inf": "{:+.1%}", "IC 95% sup": "{:+.1%}"})

,comparação,métrica,diferença,IC 95% inf,IC 95% sup,distinguível de zero
0,regra − original,F1-macro,+8.9%,+7.0%,+10.8%,True
1,ajuste fino − original,F1-macro,+19.4%,+13.5%,+25.2%,True
2,ajuste fino − regra,F1-macro,+10.6%,+4.6%,+16.8%,True
3,regra − original,acurácia,+12.2%,+9.7%,+14.7%,True
4,ajuste fino − original,acurácia,+19.2%,+15.9%,+22.4%,True
5,ajuste fino − regra,acurácia,+7.0%,+4.1%,+10.2%,True
6,regra − original,F1 negative,+0.0%,+0.0%,+0.0%,False
7,ajuste fino − original,F1 negative,+9.3%,-0.3%,+18.9%,False
8,ajuste fino − regra,F1 negative,+9.3%,-0.3%,+18.9%,False
9,regra − original,F1 positive,+9.1%,+7.4%,+10.8%,True


### 7.1.1 Robustez: só os comentários do teste cujo texto não aparece no desenvolvimento

O teste e o desenvolvimento são sorteios da mesma população e têm textos em comum (os mais frequentes, como aplausos). O modelo
não é treinado no desenvolvimento, mas a época e a taxa de aprendizado são escolhidas nele, o que poderia favorecer o ajuste fino
nos textos repetidos. A tabela repete a comparação só nos comentários do teste cujo texto normalizado **não** aparece no
desenvolvimento. Este recorte não representa a população (perde justamente os textos mais comuns); serve para ver se a conclusão
depende dos textos repetidos.

In [7]:
fora_do_dev = ~teste["text"].map(chave).isin(chaves["desenvolvimento"]).to_numpy()
print(f"Comentários do teste com texto fora do desenvolvimento: {fora_do_dev.sum()} de {len(teste)}",
      {c: int((fora_do_dev & (estrato_idx == i)).sum()) for i, c in enumerate(CLASSES)})

sub = np.flatnonzero(fora_do_dev)
pontual_sub = {m: metricas(J_de(prev_idx[m][sub], real_idx[sub], estrato_idx[sub])) for m in METODOS}
ix_sub = [sub[estrato_idx[sub] == i] for i in range(3)]
rng_sub = np.random.default_rng(am.SEED + 1)
boot_sub = {m: [] for m in METODOS}
for _ in range(4000):
    re = np.concatenate([rng_sub.choice(ix, size=len(ix), replace=True) for ix in ix_sub])
    for m in METODOS:
        boot_sub[m].append(metricas(J_de(prev_idx[m][re], real_idx[re], estrato_idx[re]))["F1-macro"])
boot_sub = {m: np.array(v) for m, v in boot_sub.items()}

linhas = [{"método": m, "acurácia": pontual_sub[m]["acurácia"], "F1-macro": pontual_sub[m]["F1-macro"],
           "F1 negative": pontual_sub[m]["F1 negative"], "acertos na amostra": int((prev_idx[m][sub] == real_idx[sub]).sum())} for m in METODOS]
display(pd.DataFrame(linhas).set_index("método").style.format({"acurácia": "{:.1%}", "F1-macro": "{:.1%}", "F1 negative": "{:.1%}"}))
for a, b in [("regra", "original"), ("ajuste fino", "original"), ("ajuste fino", "regra")]:
    d = boot_sub[a] - boot_sub[b]
    print(f"F1-macro, {a} − {b}: {pontual_sub[a]['F1-macro'] - pontual_sub[b]['F1-macro']:+.1%}  IC 95% [{np.percentile(d, 2.5):+.1%}; {np.percentile(d, 97.5):+.1%}]")

Comentários do teste com texto fora do desenvolvimento: 349 de 450 {'positive': 101, 'neutral': 118, 'negative': 130}


,acurácia,F1-macro,F1 negative,acertos na amostra
método,,,,
original,74.8%,65.9%,68.4%,234
regra,84.8%,73.0%,68.4%,272
ajuste fino,92.2%,82.3%,75.8%,300


F1-macro, regra − original: +7.1%  IC 95% [+5.2%; +9.2%]
F1-macro, ajuste fino − original: +16.4%  IC 95% [+9.5%; +23.2%]
F1-macro, ajuste fino − regra: +9.3%  IC 95% [+2.2%; +16.4%]


### 7.2 Matrizes de confusão no teste

Contagens na amostra (linhas: previsto pelo método; colunas: anotado). A amostra é estratificada pela classe do classificador
original, então estas contagens servem para ver o tipo de erro, não as proporções da população.

In [8]:
for m in METODOS:
    print(m)
    display(pd.crosstab(teste[m], teste["real"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0).rename_axis(index="previsto", columns="anotado"))

original


anotado,positive,neutral,negative
previsto,,,
positive,147,2,1
neutral,94,43,13
negative,16,22,112


regra


anotado,positive,neutral,negative
previsto,,,
positive,206,2,1
neutral,35,43,13
negative,16,22,112


ajuste fino


anotado,positive,neutral,negative
previsto,,,
positive,242,5,5
neutral,4,43,5
negative,11,19,116


### 7.3 Onde os métodos diferem

In [9]:
acerto = {m: teste[m] == teste["real"] for m in METODOS}
print("Acertos na amostra de teste:", {m: int(acerto[m].sum()) for m in METODOS}, "de", len(teste))
print("\nRegra x ajuste fino (linhas: regra acerta; colunas: ajuste fino acerta):")
display(pd.crosstab(acerto["regra"].rename("regra acerta"), acerto["ajuste fino"].rename("ajuste fino acerta")))

print("Em comentários em Código de campanha que o original chamou de neutral (onde a regra atua):")
alvo = teste[(teste["original"] == "neutral") & teste["codigo_campanha"]]
display(pd.DataFrame({m: (alvo[m] == alvo["real"]).agg(["sum", "size"]) for m in METODOS}).rename(index={"sum": "acertos", "size": "n"}))

print("Nos demais comentários que o original chamou de neutral (onde a regra não atua):")
resto = teste[(teste["original"] == "neutral") & ~teste["codigo_campanha"]]
display(pd.DataFrame({m: (resto[m] == resto["real"]).agg(["sum", "size"]) for m in METODOS}).rename(index={"sum": "acertos", "size": "n"}))

Acertos na amostra de teste: {'original': 302, 'regra': 361, 'ajuste fino': 401} de 450

Regra x ajuste fino (linhas: regra acerta; colunas: ajuste fino acerta):


ajuste fino acerta,False,True
regra acerta,,
False,32,57
True,17,344


Em comentários em Código de campanha que o original chamou de neutral (onde a regra atua):


,original,regra,ajuste fino
acertos,0,59,58
n,59,59,59


Nos demais comentários que o original chamou de neutral (onde a regra não atua):


,original,regra,ajuste fino
acertos,43,43,76
n,91,91,91


In [10]:
with pd.option_context("display.max_rows", 200):
    print("Ajuste fino acerta e regra erra:")
    display(teste.loc[acerto["ajuste fino"] & ~acerto["regra"], ["id_amostra", "text", "real", "regra", "ajuste fino"]].reset_index(drop=True))
    print("Regra acerta e ajuste fino erra:")
    display(teste.loc[acerto["regra"] & ~acerto["ajuste fino"], ["id_amostra", "text", "real", "regra", "ajuste fino"]].reset_index(drop=True))

[saída omitida na versão versionada: mostra textos reais de comentários]


## 8. Aplicação do critério

In [11]:
def vence(a, b, col="F1-macro"):
    d = boot[a][col] - boot[b][col]
    return bool(np.percentile(d, 2.5) > 0)

ordem_simplicidade = ["original", "regra", "ajuste fino"]
elegiveis = [m for m in METODOS if not vence("original", m, "F1 negative")]
escolhido = elegiveis[0]
for m in elegiveis[1:]:
    if vence(m, escolhido):
        escolhido = m
print("Elegíveis pela salvaguarda da classe negative:", elegiveis)
for a, b in pares:
    print(f"{a} vence {b} em F1-macro com diferença distinguível de zero: {vence(a, b)}")
print("\nMétodo escolhido pelo critério da Seção 6:", escolhido.upper())

registro = {
    "teste": {"n": int(len(teste)), "sha256_rotulos": hashlib.sha256(ARQ_ROTULOS_TESTE.read_bytes()).hexdigest(),
              "rotulos_conferidos": int((rev != "").sum())},
    "treino": {"n": int(len(treino_txt)), "distribuicao": rot_treino["rotulo"].value_counts().to_dict()},
    "ajuste_fino": log["escolhido"],
    "metricas": {m: {k: float(v) for k, v in pontual[m].items()} for m in METODOS},
    "ic95": {m: {k: [float(x) for x in np.percentile(boot[m][k], [2.5, 97.5])] for k in ("acurácia", "F1-macro", "F1 negative")} for m in METODOS},
    "diferencas": dif.to_dict(orient="records"),
    "proporcoes_pela_anotacao": {c: float(real_J[i]) for i, c in enumerate(CLASSES)},
    "escolhido": escolhido,
}
(PASTA / "metricas_comparacao.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")

Elegíveis pela salvaguarda da classe negative: ['original', 'regra', 'ajuste fino']
regra vence original em F1-macro com diferença distinguível de zero: True
ajuste fino vence original em F1-macro com diferença distinguível de zero: True
ajuste fino vence regra em F1-macro com diferença distinguível de zero: True

Método escolhido pelo critério da Seção 6: AJUSTE FINO


6847

## 9. Adoção do método escolhido

O ajuste fino foi adotado (ADR 0049): `scripts/reclassify_comment_sentiment.py` rodou o modelo ajustado em todos os comentários da
Gold, gravou o rótulo novo em `sentiment_label` (o do classificador original fica em `sentiment_label_modelo`) e recalculou a NSM,
o Escore composto e o Score ICE. Esta seção confere que a Gold reproduz o método avaliado no teste e mede o que a troca muda nas
métricas da Coleta. Nada aqui entrou na escolha do método.

### 9.1 A Gold reproduz o método avaliado?

In [12]:
assert "sentiment_label_ajuste" in pop.columns and (pop["sentiment_label_ajuste"] != pop["sentiment_label"]).any(), \
    "a Gold ainda não foi reclassificada: rode scripts/reclassify_comment_sentiment.py"
na_gold = teste_gab[["id_amostra", "id_comment"]].astype({"id_comment": str}).merge(
    pop[["id_comment", "sentiment_label_ajuste"]].astype({"id_comment": str}), on="id_comment", how="left")
confere = teste[["id_amostra", "ajuste fino"]].merge(na_gold, on="id_amostra")
iguais = int((confere["ajuste fino"] == confere["sentiment_label_ajuste"]).sum())
print(f"Rótulo da Gold igual à previsão avaliada no teste: {iguais} de {len(confere)}")
assert iguais >= 0.99 * len(confere)

Rótulo da Gold igual à previsão avaliada no teste: 450 de 450

### 9.2 Distribuição dos rótulos na população

In [13]:
COLUNAS = {"original": "sentiment_label", "regra": "sentiment_label_regra", "ajuste fino": "sentiment_label_ajuste"}
geral = pd.DataFrame({m: pop[c].value_counts(normalize=True).reindex(CLASSES) for m, c in COLUNAS.items()})
virais = pd.DataFrame({m: pop.loc[pop["viral"], c].value_counts(normalize=True).reindex(CLASSES) for m, c in COLUNAS.items()})
display(pd.concat({"todos os comentários": geral, "comentários de Reels virais": virais}).style.format("{:.1%}"))
print("Negativos nos virais ÷ negativos no conjunto:", {m: round(float(virais.loc["negative", m] / geral.loc["negative", m]), 1) for m in METODOS})
print(f"Comentários que mudaram de rótulo do original para o ajuste fino: {(pop['sentiment_label'] != pop['sentiment_label_ajuste']).mean():.1%}")
display(pd.crosstab(pop["sentiment_label"], pop["sentiment_label_ajuste"], rownames=["original"], colnames=["ajuste fino"]).reindex(index=CLASSES, columns=CLASSES))

Negativos nos virais ÷ negativos no conjunto: {'original': 2.7, 'regra': 2.7, 'ajuste fino': 2.9}
Comentários que mudaram de rótulo do original para o ajuste fino: 22.9%


ajuste fino,positive,neutral,negative
original,,,
positive,45226,272,297
neutral,13675,7490,2034
negative,373,466,5032


### 9.3 Efeito por perfil

Aprovação com o total de comentários do perfil no denominador, como a NSM; a NSM de cada método é a aprovação vezes o mesmo alcance
médio. Perfis com ao menos cem comentários.

In [14]:
nsm = repo.load_nsm().set_index("inputUrl")
por_perfil = pd.DataFrame({"perfil": nsm["username"], "comentários": nsm["n_comentarios_totais"]})
for m, c in COLUNAS.items():
    por_perfil[f"aprovação: {m}"] = pop[pop[c] == "positive"].groupby("inputUrl").size().reindex(nsm.index).fillna(0) / nsm["n_comentarios_totais"]
por_perfil = por_perfil[por_perfil["comentários"] >= 100].copy()
alcance_medio = nsm.loc[por_perfil.index, "alcance_medio"]
assert np.allclose(por_perfil["aprovação: ajuste fino"], nsm.loc[por_perfil.index, "proporcao_positivos"]), "a NSM da Gold não usa o rótulo ajustado"
posto = lambda s: s.rank(ascending=False).astype(int)
for m in METODOS:
    por_perfil[f"posição aprovação: {m}"] = posto(por_perfil[f"aprovação: {m}"])
    por_perfil[f"posição NSM: {m}"] = posto(por_perfil[f"aprovação: {m}"] * alcance_medio)
por_perfil["ajuste fino − original (p.p.)"] = (por_perfil["aprovação: ajuste fino"] - por_perfil["aprovação: original"]) * 100
por_perfil["ajuste fino − regra (p.p.)"] = (por_perfil["aprovação: ajuste fino"] - por_perfil["aprovação: regra"]) * 100

display(por_perfil.sort_values("ajuste fino − original (p.p.)", ascending=False).set_index("perfil").style.format(
    {"comentários": "{:,.0f}", **{f"aprovação: {m}": "{:.1%}" for m in METODOS},
     "ajuste fino − original (p.p.)": "{:+.1f}", "ajuste fino − regra (p.p.)": "{:+.1f}"}))

sp = lambda a, b: por_perfil[a].corr(por_perfil[b], method="spearman")
for base in ("original", "regra"):
    d = por_perfil[f"ajuste fino − {base} (p.p.)"]
    print(f"Ajuste fino × {base}: aprovação muda de {d.min():+.1f} a {d.max():+.1f} p.p.; "
          f"mudam de posição em aprovação {(por_perfil[f'posição aprovação: {base}'] != por_perfil['posição aprovação: ajuste fino']).sum()} de {len(por_perfil)} "
          f"(Spearman {sp(f'aprovação: {base}', 'aprovação: ajuste fino'):.2f}); "
          f"na NSM, {(por_perfil[f'posição NSM: {base}'] != por_perfil['posição NSM: ajuste fino']).sum()} de {len(por_perfil)} "
          f"(Spearman {sp(f'posição NSM: {base}', 'posição NSM: ajuste fino'):.2f})")

,comentários,aprovação: original,aprovação: regra,aprovação: ajuste fino,posição aprovação: original,posição NSM: original,posição aprovação: regra,posição NSM: regra,posição aprovação: ajuste fino,posição NSM: ajuste fino,ajuste fino − original (p.p.),ajuste fino − regra (p.p.)
perfil,,,,,,,,,,,,
wilsonlimaam,"2,093",40.5%,76.2%,83.5%,22,21,9,20,9,21,+43.0,+7.3
ricardoferraco,"3,369",46.2%,68.2%,81.9%,21,20,14,19,11,19,+35.7,+13.7
fabiogov55,"6,277",55.7%,82.8%,86.5%,15,18,1,15,5,15,+30.8,+3.7
clecioluis_,"4,845",61.4%,82.0%,86.9%,10,14,3,14,4,14,+25.5,+4.9
gladsoncameli,"2,597",63.3%,79.4%,87.1%,8,22,6,22,2,22,+23.8,+7.7
antoniodenariumrr,"2,026",63.9%,76.9%,82.6%,7,23,8,23,10,23,+18.7,+5.7
elmanofreitas,"5,395",62.7%,73.3%,81.3%,9,12,10,12,12,12,+18.6,+8.0
jeronimorodriguesba,"4,292",54.1%,61.5%,72.4%,16,4,17,4,16,4,+18.2,+10.8
ratinho_junior,"4,862",57.3%,68.9%,75.4%,13,17,13,16,13,17,+18.2,+6.6


Ajuste fino × original: aprovação muda de +1.5 a +43.0 p.p.; mudam de posição em aprovação 18 de 24 (Spearman 0.69); na NSM, 8 de 24 (Spearman 0.99)
Ajuste fino × regra: aprovação muda de +0.6 a +13.7 p.p.; mudam de posição em aprovação 17 de 24 (Spearman 0.96); na NSM, 6 de 24 (Spearman 1.00)


## 10. Leitura

> Os números desta seção são os da anotação **antes da conferência do autor** (nenhum `rotulo_revisado` preenchido em
> `teste_planilha_conferencia.xlsx`). Depois da conferência, reexecute o notebook e atualize esta leitura.

**1. O ajuste fino vence pelo critério fixado.** No teste, o F1-macro estimado para a população é de 65,7% no classificador
original (IC 95% de 61,2% a 70,1%), 74,5% com a regra (69,6% a 79,4%) e 85,1% com o ajuste fino (80,2% a 89,6%). A acurácia é de
74,7%, 86,9% e 93,9%. As três diferenças de F1-macro são distinguíveis de zero: regra sobre original, +8,9 pontos (+7,0 a +10,8);
ajuste fino sobre original, +19,4 (+13,5 a +25,2); ajuste fino sobre regra, +10,6 (+4,6 a +16,8).

**2. A regra faz o que promete, e só isso.** Nos 59 comentários do teste em Código de campanha que o original chamou de `neutral`,
a regra acerta 59 e o ajuste fino, 58. O teste é um sorteio novo, e a regra, que foi desenhada em outra amostra, manteve o
desempenho: acurácia de 86,9% aqui, contra os 85,2% estimados na amostra em que nasceu. Onde a regra não atua (os outros 91
"neutros" do original), ela acerta os mesmos 43 que o original; o ajuste fino acerta 76.

**3. O ganho do ajuste fino está no que a regra não alcança.** Apoio em texto corrido ("Bora Lula", "Tô com vc", "Tem meu voto"),
número de candidato com marcador verbal ("Pra cima 44", "5️⃣5️⃣ minha governadora") e crítica que o original lia como neutra
("Você nunca mais", cobranças em forma de pergunta). A precisão de `neutral` vai de 28,7% (original) a 47,3% (regra) e 86,3%
(ajuste fino).

**4. A classe `negative` não piora, e provavelmente melhora.** O F1 de `negative` é de 69,7% no original e na regra e de 79,0% no
ajuste fino; a diferença de +9,3 pontos tem intervalo de −0,3 a +18,9, isto é, não chega a ser distinguível de zero. A revocação
sobe de 65,4% para 85,0%, com precisão igual (74,7% e 73,8%). O ajuste fino prevê 10,3% de negativos, contra 8,9% estimados pela
anotação e 7,8% do original: ele encontra mais críticas e passa um pouco do ponto.

**5. A aprovação deixa de ser subestimada.** A anotação estima 80,2% de positivos. O original prevê 61,2%, a regra 73,4% e o
ajuste fino 80,6%.

**6. A conclusão não depende dos textos repetidos entre teste e desenvolvimento.** Só nos 349 comentários do teste cujo texto não
aparece no desenvolvimento, o F1-macro é de 65,9%, 73,0% e 82,3%, e o ajuste fino segue acima da regra com diferença
distinguível (+9,3 pontos, de +2,2 a +16,4).

**7. Onde o ajuste fino erra e a regra acerta (17 comentários).** Erros de três tipos: insulto curto ou grafado de forma incomum
lido como neutro ou positivo ("Escravista", "L@draoooooooo"), texto neutro puxado para um lado ("Já votei" como positivo, "onde é
esse hospital?" como negativo) e um Código de campanha muito repetido que saiu neutro. São erros sem padrão único, ao contrário do
erro do original.

**8. Adoção (Seção 9).** O ajuste fino foi adotado na Gold, e o rótulo gravado coincide com a previsão avaliada no teste em 450 de
450 comentários. Na população, 22,9% dos comentários mudam de rótulo em relação ao original, quase todos de `neutral` para
`positive`. A aprovação por perfil sobe de 1,5 a 43,0 pontos; 18 dos 24 perfis mudam de posição em aprovação (correlação de postos
de 0,69 com o original e de 0,96 com a regra). Na NSM, que multiplica a aprovação pelo alcance médio, só 8 dos 24 mudam de posição
(correlação de 0,99). O achado dos virais não depende do método: a razão entre negativos nos virais e no conjunto é de 2,7 no
original e na regra e de 2,9 no ajuste fino.

**Ressalva principal: o ajuste fino aprendeu o critério de quem anotou o teste.** O treino foi anotado por instâncias do mesmo
modelo de linguagem que anotou o teste, com o mesmo critério escrito. Não há vazamento de dados (nenhum texto nem publicação do
teste está no treino), mas há um anotador em comum: parte da vantagem do ajuste fino pode ser concordância com o estilo desse
anotador, e não com o julgamento de uma pessoa. A regra e o original não têm essa vantagem. A conferência do autor na planilha do
teste é o que mede isso: se os rótulos conferidos mantiverem a ordem dos três métodos, a ressalva perde força.

**Outras limitações.** Uma única semente de treino e duas taxas de aprendizado; não foi medida a variação entre execuções. O
treino tem 1.482 textos de uma Coleta em período eleitoral. O modelo foi ajustado em comentários; legendas não fizeram parte do
treino nem do teste. O teste tem 450 comentários, e as métricas por classe têm intervalos de dez a vinte pontos.